## Implementation of Gradio GUI for RAG-Chatbot (RAG-Fusion + In-memory persistence)

### Libraries, ChatOllama, Chroma vectorstore initialization

In [1]:
from chromadb.config import Settings
from chromadb import Client
from langchain.vectorstores import Chroma
import chromadb

from langchain_ollama import ChatOllama
from langchain_community.embeddings import OllamaEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from typing_extensions import List, TypedDict
from langgraph.graph import START, StateGraph
from langgraph.checkpoint.memory import MemorySaver

from datetime import datetime
import os, uuid, gradio as gr

date = datetime.today().strftime('%Y-%m-%d')

# Pipeline Switch
USE_COLBERT = True
proj_name = f"Gradio GUI ({'ColBERT' if USE_COLBERT else 'RAG Fusion'} + Memory Persistence)"
run_count = 1

# Initialize Langsmith
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = "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"
os.environ["LANGSMITH_PROJECT"] = f"[{date}] VAA - {proj_name} {run_count}"

# Initialize LLM (Temperature set to 0.5 for more focused responses)
llm = ChatOllama(model="deepseek-r1:8b", validate_model_on_init=True, temperature=0.5, reasoning=True) 
simpler_llm = ChatOllama(model="deepseek-r1:8b", validate_model_on_init=True, reasoning=False)
emb = OllamaEmbeddings(model="bge-m3:567m")

num_queries = 4     # Number of additional queries to generate in RAG-Fusion
num_docs = 7        # Number of top documents to select
num_chat_his = 3    # Number of previous chat messages to include in context

/var/folders/cf/1j9rc9v11rzf5wxcjw3w_wsm0000gp/T/ipykernel_51134/1621653536.py:33: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  emb = OllamaEmbeddings(model="bge-m3:567m")


In [2]:
SINGLE = True # Change to True if you want to use single chroma database for all documents
collection_name = "academic_documents" if not SINGLE else "vaa_documents"

# Initialize retriever for queries. Get the workspace root directory
import pathlib

def get_project_root() -> pathlib.Path:
    current_file_dir = pathlib.Path(pathlib.Path.cwd()).resolve().parent
    if (current_file_dir / '.git').exists():
        return current_file_dir
    for parent in current_file_dir.parents:
        if (parent / '.git').exists():
            return parent
    return pathlib.Path.cwd() # Fallback to current working directory if .git not found

ROOT_DIR = get_project_root()

chroma_db_path = ROOT_DIR / "chroma_db"
print(f"Chroma DB path: {chroma_db_path}")

client = Client(Settings())
client = chromadb.PersistentClient(path=str(chroma_db_path))

vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=emb
)

client.get_collection(name=collection_name).count()

/var/folders/cf/1j9rc9v11rzf5wxcjw3w_wsm0000gp/T/ipykernel_51134/1086128921.py:24: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorStore = Chroma(


Chroma DB path: /Users/MarcussPC/Desktop/Temp/CAPSTONE/chroma_db


4320

### ColBERT Initialization

In [3]:
if USE_COLBERT:
    import platform, multiprocessing
    
    print(f"Processor: {platform.processor()}")
    print(f"Machine: {platform.machine()}")
    
    try:
        multiprocessing.set_start_method("spawn", force=True)
    except RuntimeError:
        pass
    
    # Disable CUDA to prevent the kernel from looking for NVIDIA drivers
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
    os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
    
    from ragatouille import RAGPretrainedModel
    
    # Initialize ColBERT model
    colbert = RAGPretrainedModel.from_pretrained("colbert-ir/colbertv2.0")
    print("ColBERT model loaded successfully")
else:
    print("Using RAG-Fusion pipeline (ColBERT disabled)")

/var/folders/cf/1j9rc9v11rzf5wxcjw3w_wsm0000gp/T/ipykernel_51134/3375193420.py:16: UserWarning: 
********************************************************************************
RAGatouille WARNING: Future Release Notice
--------------------------------------------
RAGatouille version 0.0.10 will be migrating to a PyLate backend 
instead of the current Stanford ColBERT backend.
PyLate is a fully mature, feature-equivalent backend, that greatly facilitates compatibility.
However, please pin version <0.0.10 if you require the Stanford ColBERT backend.
********************************************************************************
  from ragatouille import RAGPretrainedModel


Processor: arm
Machine: arm64
[Feb 13, 15:51:23] Loading segmented_maxsim_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...
ColBERT model loaded successfully


/opt/anaconda3/envs/FYPEnv/lib/python3.12/site-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
/opt/anaconda3/envs/FYPEnv/lib/python3.12/site-packages/torch/amp/grad_scaler.py:136: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


### All Prompts for the Pipeline

In [4]:
# ================= Query Classification prompt =================
QUERY_CLASSIFICATION_PROMPT = \
"""
You are a helpful assistant of The Hong Kong Polytechnic University (PolyU). Please classify the student's query into one of the two categories:

1. 'DOMAIN': The question is about university-related information, including academics, subjects, student life, university services, facilities, career, or internship guidance.
3. 'OFFTOPIC': The question completely unrelated to the university, including but not limited to a greeting or a general knowledge.

*Do NOT answer the question*, just return the category name: 'DOMAIN' or 'OFFTOPIC'.

*User Question*:
{question}

Category:
"""
query_classification_prompt = PromptTemplate.from_template(QUERY_CLASSIFICATION_PROMPT)

# ================= Chatting Response prompt =================
OFF_TOPIC_PROMPT = \
"""
You are a professional academic advisor at the Department of Electrical and Electronic Engineering (EEE) of The Hong Kong Polytechnic University (PolyU). Given the following information:

======
*Previous Conversation*:
{chat_history}
======
*Student's Question*:
{question}
======

Respond warmly to the user's message. Keep it brief and helpful.
Politely inform the user that their question is outside your area of expertise.
Redirect them to ask about academics, subjects, student services, or university information instead.

Response:
"""
offtopic_prompt = PromptTemplate.from_template(OFF_TOPIC_PROMPT)


# ================= Query Rewriting prompt =================
TRANSFORM_QUERY_PROMPT = \
"""
Given a chat history and the latest user question which might reference context in the chat history:

======
*Chat History*:
{chat_history}
======

Now, formulate a standalone question which can be understood without the chat history. 
*Do NOT answer the question*, just reformulate it if needed and otherwise return it as is.

*User Question*:
{question}

Reformulated standalone question:
"""
query_tra_prompt = PromptTemplate.from_template(TRANSFORM_QUERY_PROMPT)

# ================= RAG-Fusion prompt =================
RAG_FUSION_PROMPT = \
"""
You are a helpful assistant that generates multiple alternative queries based on a single input query. 
If the input query contains multiple sub-questions, ensure each alternative question focuses on a specific sub-question.

Provide strictly {num_queries} alternative questions separated by newlines. Do not say anything else.

*User Question*: 
{question}

{num_queries} alternative questions:
"""
query_gen_prompt = PromptTemplate.from_template(RAG_FUSION_PROMPT)

# ================= DeepSeek-R1 prompt =================
LLM_PROMPT = \
"""
You are a professional academic advisor at the Department of Electrical and Electronic Engineering (EEE) of The Hong Kong Polytechnic University (PolyU). Given the following information:

======
*Previous Conversation*:
{chat_history}
======
*Context*:
{context}
======
*Student's Question*:
{question}
======

Please adhere to the following rules when answering the student's question:
1. If the user is asking non-academic questions, please politely inform them your roles and encourage them to ask academic questions.
2. Use the information from the previous conversation first, then the context, to answer the student's question.
3. Answer in the same language as the user query, e.g., English query, English answer.
4. Avoid saying "may", "maybe", or similar; be affirmative, confident, and decisive in your answers.
5. Avoid saying "based on the provided context", or similar; answer directly.
6. Say no if you cannot answer the question; *never fabricate a factually false answer*. Instead, ask for clarification.
7. Provide relevant URLs if necessary. However, *never fabricate non-existence URLs*. Only provide URLs from the context.
8. Provide advice to the student based on your answer and ask for any further enquiries, if applicable.

Now, give a helpful answer to the student!
"""
prompt = PromptTemplate.from_template(LLM_PROMPT)

### Main RAG Pipeline

In [5]:
# Initialize the memory saver
memory = MemorySaver()

# State class
class State(TypedDict, total=False):
    question: str                       # Required: user's question
    query_type: str                     # Classification: 'DOMAIN' or 'OFFTOPIC'
    contextualized_question: str        # Rewritten query

    queries: List[str]                  # Generated queries for RAG-Fusion
    context: List[Document]             # Retrieved documents

    prepared_messages: List[dict]       # Store prepared messages for streaming
    prepared_question: str              # Store the question used for generation

    answer: str                         # Generated answer (from streaming)   
    chat_history: List[dict]            # Store chat history as list of message dicts

def classify_query(state: State):
    question = state["question"]
    
    messages = query_classification_prompt.invoke({"question": question})
    response = simpler_llm.invoke(messages)
    
    if "DOMAIN" in response.content:
        query_type = "DOMAIN"
    else:
        query_type = "OFFTOPIC"
    
    print(f"[Router] Query classified as: {query_type}")
    return {"query_type": query_type}

def route_query(state: State):
    query_type = state.get("query_type", "DOMAIN")
    
    if query_type == "OFFTOPIC":
        return "OFFTOPIC"
    else:
        return "DOMAIN"

def contextualize_question(state: State):
    question = state["question"]
    chat_history = state.get("chat_history", [])
    
    # If no chat history, the question is already standalone
    if not chat_history:
        print(f"[Query Transform] No history - using original: {question}")
        return {"contextualized_question": question}
    
    # Else, convert chat history to LangChain message format
    history = ""
    for msg in chat_history[-num_chat_his:]:
        history += f"{msg['role'].capitalize()}: {msg['content']}\n\n"
    
    # Contextualize the question
    messages = query_tra_prompt.invoke({"chat_history": history, "question": question})
    response = simpler_llm.invoke(messages)
    
    contextualized_question = response.content.strip()
    
    print(f"[Query Transform] Original: {question}")
    print(f"[Query Transform] Contextualized: {contextualized_question}")
    
    return {"contextualized_question": contextualized_question}

# ====== RAG-FUSION PART ======
def generate_queries(state: State):
    """Generate multiple queries using RAG-Fusion"""
    question = state.get("contextualized_question", state["question"])
    
    messages = query_gen_prompt.invoke({"question": question, "num_queries": num_queries})
    response = simpler_llm.invoke(messages)

    queries = response.content.strip().split("\n")
    queries = [q for q in queries if q.strip() != ""]
    
    print(f"[RRF] Generated {len(queries)} queries from a question")
    
    return {"queries": queries}

def retrieve_ragfusion(state: State):
    """Retrieve documents using all generated queries"""
    all_docs = []
    
    print(f"[RRF] Retrieving for {len(state['queries'])} queries...")
    for idx, query in enumerate(state["queries"], 1):
        print(f"  Query {idx}: {query[:80]}...")
        retrieved_with_scores = vectorStore.similarity_search_with_score(query, k=4)
        
        retrieved_docs = [doc for doc, score in retrieved_with_scores]
        scores = [score for doc, score in retrieved_with_scores]
        
        if scores:
            avg_score = sum(scores) / len(scores)
            print(f"    Scores: {[f'{s:.4f}' for s in scores]}")
            print(f"    Average Score: {avg_score:.4f}")
        
        all_docs.append(retrieved_docs)
    
    return {"context": all_docs}

def rrf_ragfusion(state: State, k: int = 60):
    """Fuse and rerank documents using Reciprocal Rank Fusion (RRF)"""
    fused_scores = {}
    
    # Each query's retrieved documents contribute to the fused score based on their rank
    for docs in state["context"]:
        for rank, doc in enumerate(docs):
            doc_str = doc.page_content
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            fused_scores[doc_str] += 1 / (rank + k)
    
    reranked_results = [
        (doc_str, score) for doc_str, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]
    
    unique_docs = {doc.page_content: doc for docs in state["context"] for doc in docs}.values()
    doc_map = {doc.page_content: doc for doc in unique_docs}
    reranked_docs = [doc_map[doc_str] for doc_str, _ in reranked_results[:num_docs]]
    
    print(f"[RRF] Selected top {len(reranked_docs)} documents after fusion")
    return {"context": reranked_docs}
# ====== END ======

# ====== COLBERT PART ======
def retrieve_colbert(state: State):
    """Retrieve documents with ColBERT reranking"""
    question = state.get("contextualized_question", state["question"])
    initial_with_scores = vectorStore.similarity_search_with_score(question, k=20)
    
    docs = [doc for doc, score in initial_with_scores]
    scores = [score for doc, score in initial_with_scores]
    doc_texts = [doc.page_content for doc in docs]
    
    if scores:
        avg_initial_score = sum(scores) / len(scores)
        print(f"[ColBERT] Retrieved {len(docs)} initial candidates")
        print(f"[ColBERT] Scores: min={min(scores):.4f}, max={max(scores):.4f}, avg={avg_initial_score:.4f}")
    
    # Rerank using ColBERT
    reranked_results = colbert.rerank(
        query=question,
        documents=doc_texts,
        k=num_docs
    )
    
    # Reconstruct Document objects with reranked results
    retrieved_docs = []
    for result in reranked_results:
        for doc in docs:
            if doc.page_content == result['content']:
                retrieved_docs.append(doc)
                break
    
    print(f"[ColBERT] Reranked to top {len(retrieved_docs)} documents")
    
    return {"context": retrieved_docs}
# ====== END ======

def prompt_prepare(state: State):
    """Prepare prompt for answer generation"""
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    
    # Format chat history for the prompt
    chat_history_str = ""
    if state.get("chat_history"):
        for msg in state["chat_history"][-3:]:  # Include last 3 messages
            chat_history_str += f"{msg['role'].capitalize()}: {msg['content']}\n\n"
    else:
        chat_history_str = "No previous conversation."
    
    messages = prompt.invoke({
        "question": state.get("contextualized_question", state["question"]),
        "context": docs_content,
        "chat_history": chat_history_str
    })
    
    # Store prepared messages and question for streaming
    return {
        "prepared_messages": messages,
        "prepared_question": state.get("contextualized_question", state["question"])
    }

# ====== NON-DOMAIN HANDLERS ======
def handle_offtopic(state: State):
    """Handle greetings and casual conversation without retrieval"""
    question = state["question"]
    chat_history = state.get("chat_history", [])
    
    # Format chat history
    history_str = ""
    if chat_history:
        for msg in chat_history[-num_chat_his:]:
            history_str += f"{msg['role'].capitalize()}: {msg['content']}\n\n"
    else:
        history_str = "No previous conversation."
    
    messages = offtopic_prompt.invoke({"question": question, "chat_history": history_str})
    
    return {
        "prepared_messages": messages,
        "prepared_question": question
    }
# ====== END ======

# Build graph
def build_graph():
    """Build graph with routing and memory checkpointing"""
    graph_builder = StateGraph(State)
    
    # Add routing nodes
    graph_builder.add_node("classify_query", classify_query)
    graph_builder.add_node("handle_offtopic", handle_offtopic)

    # Add existing nodes
    graph_builder.add_node("contextualize_question", contextualize_question)
    graph_builder.add_node("prompt_prepare", prompt_prepare)
    
    if USE_COLBERT:
        graph_builder.add_node("retrieve_colbert", retrieve_colbert)
    else:
        graph_builder.add_node("generate_queries", generate_queries)
        graph_builder.add_node("retrieve", retrieve_ragfusion)
        graph_builder.add_node("fuse_and_rerank", rrf_ragfusion)
    
    # Routing from START
    graph_builder.add_edge(START, "classify_query")
    graph_builder.add_conditional_edges(
        "classify_query",
        route_query,
        {
            "DOMAIN": "contextualize_question",
            "OFFTOPIC": "handle_offtopic",
        }
    ) 
    
    # Domain-specific
    if USE_COLBERT:
        graph_builder.add_edge("contextualize_question", "retrieve_colbert")
        graph_builder.add_edge("retrieve_colbert", "prompt_prepare")
    else:
        graph_builder.add_edge("contextualize_question", "generate_queries")
        graph_builder.add_edge("generate_queries", "retrieve")
        graph_builder.add_edge("retrieve", "fuse_and_rerank")
        graph_builder.add_edge("fuse_and_rerank", "prompt_prepare")
    
    # Compile with memory checkpointer
    graph = graph_builder.compile(checkpointer=memory)
    return graph

# Recreate the graph with routing
graph = build_graph()
print("Flow:")
print("  1. Classify Query")
print("  2a. Domain         → RAG-Fusion or ColBERT")
print("  2b. OffTopic       → Direct Response")

Flow:
  1. Classify Query
  2a. Domain         → RAG-Fusion or ColBERT
  2b. OffTopic       → Direct Response


### Chat Memory Management

In [6]:
# Memory Management
def create_new_thread():
    return str(uuid.uuid4())

def get_chat_history(thread_id: str):
    """Retrieve chat history for a specific thread"""
    try:
        # Get the state from the checkpoint
        config = {"configurable": {"thread_id": thread_id}}
        state = graph.get_state(config)
        
        if state and state.values.get("chat_history"):
            return state.values["chat_history"]
        else:
            return []
    except Exception as e:
        print(f"Error retrieving chat history: {e}")
        return []

def display_chat_history(thread_id: str):
    """Display formatted chat history"""
    history = get_chat_history(thread_id)
    
    if not history:
        print(f"No chat history found for thread: {thread_id}")
        return
    
    print(f"\n{'='*60}")
    print(f"Chat History for Thread: {thread_id}")
    print(f"{'='*60}\n")
    
    for idx, msg in enumerate(history, 1):
        role = msg["role"].upper()
        content = msg["content"]
        print(f"{idx}. [{role}]")
        print(f"   {content[:200]}..." if len(content) > 200 else f"   {content}")
        print()

def clear_thread_memory(thread_id: str):
    """Clear all memory"""
    config = {"configurable": {"thread_id": thread_id}}
    
    try:
        # Update state with empty history
        graph.update_state(
            config,
            {"chat_history": []}
        )
        print(f"Cleared memory for thread: {thread_id}")
        return True
    except Exception as e:
        print(f"Error clearing memory: {e}")
        return False

### Gradio 5 Demo (Streaming)

In [7]:
# Note: need to yield a tuple of (msg + thread_id) to prevent error in unpack
def stream_response(message, history, thread_id):
    if not thread_id:
        thread_id = create_new_thread()
    config = {"configurable": {"thread_id": thread_id}}
    
    yield ("Query received. Processing...", thread_id)
    
    # Run the graph to prepare the prompt
    status = None
    for result in graph.stream({"question": message}, config=config):
        if "prompt_prepare" in result:
            status = result["prompt_prepare"]
        elif "handle_offtopic" in result:
            status = result["handle_offtopic"]
    
    # Stream the LLM response
    partial = ""
    for chunk in llm.stream(status["prepared_messages"]):
        if partial == "" and chunk.content.strip() == "":
            yield ("Thinking...", thread_id)
            continue
        partial += chunk.content
        yield (partial, thread_id)
    
    # Update chat history with the streamed answer
    current_state = graph.get_state(config)
    current_history = current_state.values.get("chat_history", [])
    
    new_history = current_history.copy()
    new_history.append({"role": "user", "content": status.get("prepared_question", message)})
    new_history.append({"role": "assistant", "content": partial})
    
    graph.update_state(config, {"chat_history": new_history, "answer": partial}, as_node="prompt_prepare")
    
    yield (partial, thread_id)

thread_state = gr.State()
demo_interface = gr.ChatInterface(
    stream_response,
    additional_outputs=[thread_state],
    additional_inputs=[thread_state],
    textbox=gr.Textbox(placeholder="Send to the LLM...", container=False, autoscroll=True, scale=7),
)
demo_interface.launch(debug=True)

/opt/anaconda3/envs/FYPEnv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


[Router] Query classified as: OFFTOPIC
[Router] Query classified as: DOMAIN
[Query Transform] Original: How can I apply for residential hall at PolyU?
[Query Transform] Contextualized: How can one apply for a residential hall at The Hong Kong Polytechnic University (PolyU)?
[ColBERT] Retrieved 20 initial candidates
[ColBERT] Scores: min=358.7217, max=416.6141, avg=401.3584


/opt/anaconda3/envs/FYPEnv/lib/python3.12/site-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()
/opt/anaconda3/envs/FYPEnv/lib/python3.12/site-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
100%|██████████| 1/1 [00:01<00:00,  1.38s/it]


[ColBERT] Reranked to top 7 documents
Keyboard interruption in main thread... closing server.
